## Objective for this Notebook    
* Build a Neural Network
* Compute Weighted Sum at Each Node
* Compute Node Activation
* Use Forward Propagation to Propagate Data



In [1]:
import numpy as np

weights = np.around(np.random.uniform(size=6), decimals=2)
biases = np.around(np.random.uniform(size=3), decimals=2)

In [2]:
print(weights)
print(biases)

[0.09 0.22 0.06 0.36 0.32 0.8 ]
[0.09 0.63 0.35]


In [3]:
x_1 = 0.5
x_2 = 0.85 

In [12]:
z_11 = x_1 * weights[0] + x_2 * weights[1] + biases[0]
print('The weighted sum of the inputs at the first node in the hidden layer is {}'.format(np.around(z_11, decimals =4)))

The weighted sum of the inputs at the first node in the hidden layer is 0.322


In [13]:
z_12 = x_1 * weights[2] + x_2 * weights[3] + biases[1]
print('The weighted sum of the inputs at the second node in the hidden layer is {}'.format(np.around(z_12, decimals=4)))

The weighted sum of the inputs at the second node in the hidden layer is 0.966


With Sigmoid activation function, let's compute the activation of the first node in the hidden layer.


In [15]:
a_11 = 1.0 / (1.0 + np.exp(-z_11))
print('The activation of the first node in the hidden layer is {}'.format(np.around(a_11, decimals=4)))

The activation of the first node in the hidden layer is 0.5798


In [16]:
a_12 = 1.0 / (1.0 + np.exp(-z_12))
print('The activation of the first node in the hidden layer is {}'.format(np.around(a_12, decimals=4)))

The activation of the first node in the hidden layer is 0.7243


Now these activations will serve as the inputs to the output layer. So, let's compute the weighted sum of these inputs to the node in the output layer. Assign the value to **z**.


In [17]:
z = a_11 * weights[4] + a_12 * weights[5] + biases[2]
print('The weighted sum of the inputs at the node in the output layer is {}'.format(np.around(z, decimals=4)))

The weighted sum of the inputs at the node in the output layer is 1.115


Finally, let's compute the output of the network as the activation of the node in the output layer. Assign the value to **a**.


In [19]:
a = 1.0 / (1.0 + np.exp(-z))
print('The output of the network for x1 = 0.5 and x2 = 0.85 is {}'.format(np.around(a, decimals=4)))

The output of the network for x1 = 0.5 and x2 = 0.85 is 0.7531


Obviously, neural networks for real problems are composed of many hidden layers and many more nodes in each layer. So, we can't continue making predictions using this very inefficient approach of computing the weighted sum at each node and the activation of each node manually. 


## Build a Neural Network


In [37]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1 - a)

In [40]:
def initialize_network(n_inputs, n_hidden, n_outputs):
    network = {
        "hidden": {
            "weights": np.random.rand(n_hidden, n_inputs),
            "bias": np.random.rand(n_hidden, 1)
        },
        "output": {
            "weights": np.random.rand(n_outputs, n_hidden),
            "bias": np.random.rand(n_outputs, 1)
        }
    }
    return network

In [42]:
def forward_propagate(network, X):
    # hidden layer
    z_hidden = np.dot(network["hidden"]["weights"], X) + network["hidden"]["bias"]
    a_hidden = sigmoid(z_hidden)

    # output layer
    z_output = np.dot(network["output"]["weights"], a_hidden) + network["output"]["bias"]
    a_output = sigmoid(z_output)

    cache = {
        "X": X,
        "z_hidden": z_hidden,
        "a_hidden": a_hidden,
        "z_output": z_output,
        "a_output": a_output
    }

    return a_output, cache

In [43]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [46]:
def backward_propagate(network, cache, y_true, learning_rate):
    # output layer error
    output_error = cache["a_output"] - y_true
    output_delta = output_error * sigmoid_derivative(cache["a_output"])

    # hidden layer error
    hidden_error = np.dot(network["output"]["weights"].T, output_delta)
    hidden_delta = hidden_error * sigmoid_derivative(cache["a_hidden"])

    # update output weights & bias
    network["output"]["weights"] -= learning_rate * np.dot(output_delta, cache["a_hidden"].T)
    network["output"]["bias"] -= learning_rate * output_delta

    # update hidden weights & bias
    network["hidden"]["weights"] -= learning_rate * np.dot(hidden_delta, cache["X"].T)
    network["hidden"]["bias"] -= learning_rate * hidden_delta

In [48]:
def train(network, X, y, epochs=1000, lr=0.1):
    
    for epoch in range(epochs):
        y_pred, cache = forward_propagate(network, X)
        loss = mse(y, y_pred)
        backward_propagate(network, cache, y, lr)

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Loss: {loss:.4f}")

In [52]:
X = np.array([[1.0],
              [0.5]])

y = np.array([[1.0]])

network = initialize_network(
    n_inputs=2,
    n_hidden=3,
    n_outputs=1
)
train(network, X, y, epochs=1000, lr=0.1)
prediction, _ = forward_propagate(network, X)
print("Final prediction:", prediction)

Epoch 0 | Loss: 0.0149
Epoch 100 | Loss: 0.0087
Epoch 200 | Loss: 0.0060
Epoch 300 | Loss: 0.0046
Epoch 400 | Loss: 0.0037
Epoch 500 | Loss: 0.0031
Epoch 600 | Loss: 0.0026
Epoch 700 | Loss: 0.0023
Epoch 800 | Loss: 0.0020
Epoch 900 | Loss: 0.0018
Final prediction: [[0.95938244]]
